<a href="https://colab.research.google.com/github/mohamadfaisalbashir/grokking-Deep-Learning/blob/main/06_building_your_first_deep_neural_network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Chapter 6: Building Your First Deep Neural Network - Introduction to Backpropagation**

This notebook summarizes Chapter 6 of *Grokking Deep Learning*:

1. The Streetlight Problem
2. Preparing the Data
3. Matrices and the Matrix Relationship
4. Creating a Matrix or Two in Python
5. Building a Neural Network
6. Learning the Whole Dataset
7. Full, Batch, and Stochastic Gradient Descent
8. Neural Networks Learn Correlation
9. Up and Down Pressure
10. Edge Case: Overfitting
11. Edge Case: Conflicting Pressure
12. Learning Indirect Correlation
13. Backpropagation: Long-Distance Error Attribution
14. Linear vs. Nonlinear
15. The Secret to Sometimes Correlation
16. A Quick Break
17. Your First Deep Neural Network
18. Backpropagation in Code
19. Putting It All Together
20. Why Do Deep Networks Matter?
21. Summary

The previous chapters trained small networks on **a single training example** at a time. This chapter moves to **entire datasets** by means of a toy problem, the *streetlight problem*. Along the way it explains what a network actually learns (**correlation**), what can go wrong (**overfitting** and **conflicting pressure**), and why some datasets cannot be solved by two layers at all. The remedy is a **deep network**: a stack of layers that creates an *intermediate dataset* with usable correlation, trained with **backpropagation** and a simple **nonlinearity** called `relu`. The chapter ends with the complete code of a three-layer network and a short explanation of why such networks matter.

The chapter opens with a line from Douglas Adams's *The Hitchhiker's Guide to the Galaxy*, in which a character addresses the computer Deep Thought and announces the task it has been built for: to tell everybody "The Answer".

---
## **1. The Streetlight Problem**

**This toy problem considers how a network learns entire datasets.**

Imagine arriving at a street corner in a foreign country and finding that the street light looks unfamiliar. It has three lights, and you do not know which combinations mean "walk" and which mean "stop". A reasonable strategy is to sit at the corner for a few minutes and watch the **correlation** between each light combination and what the people around you decide to do.

> An unfamiliar street light with three lights (left, middle, right), followed by the first three observations: stop, walk, stop
>
> <img src="pictures/chapter 6/picture 6.1.png" alt="Figure 6.1" width="600"/>

The first three observations are processed as follows:

1. **Left and right lights on: nobody walks.** From one datapoint alone, almost anything is possible. The left or the right light might correlate with stopping, or the central light might correlate with walking.
2. **Middle and right lights on: people walk.** Something about this light changed the signal. The only thing that can be said with confidence is that the far-right light does not seem to matter, since it was on in both cases.
3. **Only the right light on: people stop.** Now only the middle light has changed, and the outcome is the opposite one. The working hypothesis is that the **middle light indicates when it is safe to walk**.

After a few more minutes, six light patterns have been recorded together with the decision of the crowd. In the order of the figure, the outcomes are stop, walk, stop, walk, walk, and stop.

> Six recorded light patterns and whether people chose to stop or to walk
>
> <img src="pictures/chapter 6/picture 6.2.png" alt="Figure 6.2" width="600"/>

As hypothesized, there is a **perfect correlation** between the middle (crisscross) light and whether it is safe to walk. This pattern was found by observing all the individual datapoints and *searching for correlation*. That is exactly what the chapter trains a neural network to do.

---
## **2. Preparing the Data**

**Neural networks don't read streetlights.**

The earlier chapters introduced **supervised algorithms**: they take one dataset and turn it into another. More importantly, they can turn a dataset of *what you know* into a dataset of *what you want to know*. Training such a network means presenting two datasets and asking it to learn how to transform one into the other.

The streetlight problem contains exactly two datasets:

- **What you know:** the six streetlight states.
- **What you want to know:** the six observations of whether people walked.

The network should learn to convert the first dataset into the second. In the real-world situation, the state of the light is always available, and the unknown is whether crossing the street is safe. To prepare the data, it must therefore be split into these two groups. The roles could also be swapped, so that the network goes "backward"; for some problems this works as well.

> The two datasets: the streetlight states (what you know) and the observed decisions (what you want to know)
>
> <img src="pictures/chapter 6/picture 6.3.png" alt="Figure 6.3" width="600"/>


---
## **3. Matrices and the Matrix Relationship**

**Translate the streetlight into math.**

A computer does not understand streetlights, so the *pattern* of the lights must be imitated with numbers. Every light gets one **column** (three columns for three lights), and every observed streetlight gets one **row** (six rows for six observations). This structure of 1s and 0s is called a **matrix**.

> Each streetlight is mimicked by a row of three numbers; a 1 means the light is on
>
> <img src="pictures/chapter 6/picture 6.4.png" alt="Figure 6.4" width="600"/>

The usual convention for data matrices is that **each recorded example occupies one row** and **each thing being recorded occupies one column**. A column therefore holds every state in which one particular light was recorded, and a row holds the simultaneous state of all lights at one moment.

### **3.1 Good Data Matrices Perfectly Mimic the Outside World**

A data matrix does not have to consist of 1s and 0s. If the lights were on dimmers and could burn at different intensities, the matrix might look like matrix **A** below. It is perfectly valid, because it mimics the pattern of the real world.

> Streetlight matrix A: lights on dimmers, with intensities between 0 and 1
>
> <img src="pictures/chapter 6/picture 6.5.png" alt="Figure 6.5" width="600"/>

Matrix **B** is also valid. It is simply matrix A multiplied by 10 (`A * 10 == B`), so the two matrices are **scalar multiples** of each other, and both capture the relationships between the training examples (rows) and the lights (columns).

> Streetlight matrix B: matrix A multiplied by 10
>
> <img src="pictures/chapter 6/picture 6.6.png" alt="Figure 6.6" width="600"/>


### **3.2 Matrices A and B Contain the Same Underlying Pattern**

Infinitely many matrices reflect the streetlight patterns perfectly; matrix **C**, shown next, is one more of them.

> Streetlight matrix C: yet another valid matrix for the same streetlights
>
> <img src="pictures/chapter 6/picture 6.7.png" alt="Figure 6.7" width="600"/>

It is important to realize that the **underlying pattern is not the same as the matrix**. The pattern is a *property of* the matrix: it is a property of A, B, and C alike, and it already existed in the streetlights. Each matrix is merely *expressing* it.

This **input data pattern** is what the network should learn to transform into the **output data pattern**. To make this possible, the output pattern must also be captured in a matrix: stop becomes 0 and walk becomes 1. The 1s and 0s could also be reversed, since it is still possible to decode them back to stop/walk. This is why the result is called a **lossless representation**: one can convert back and forth between the original notes and the matrix without losing anything.

> The output pattern as a matrix: STOP is 0 and WALK is 1
>
> <img src="pictures/chapter 6/picture 6.8.png" alt="Figure 6.8" width="600"/>


---
## **4. Creating a Matrix or Two in Python**

**Import the matrices into Python.**

NumPy, the library introduced in Chapter 3, is made for handling matrices. A matrix is nothing more than a list of lists, an array of arrays, and NumPy is essentially a convenient wrapper around such a structure that adds matrix-oriented functions. The first cell creates the matrix of the streetlights (with 1s and 0s):

In [1]:
import numpy as np
streetlights = np.array( [ [ 1, 0, 1 ],
                           [ 0, 1, 1 ],
                           [ 0, 0, 1 ],
                           [ 1, 1, 1 ],
                           [ 0, 1, 1 ],
                           [ 1, 0, 1 ] ] )

Next comes the matrix of the output data. (The book's typesetting prints this name as `walk _ vs _ stop`, with spaces around the underscores; the cell uses the valid identifier `walk_vs_stop`.)

In [2]:
walk_vs_stop = np.array( [ [ 0 ],
                           [ 1 ],
                           [ 0 ],
                           [ 1 ],
                           [ 1 ],
                           [ 0 ] ] )

The network should take the `streetlights` matrix and learn to transform it into `walk_vs_stop`. Even more important, it should be able to take **any matrix that contains the same underlying pattern** as `streetlights` and transform it into a matrix that contains the underlying pattern of `walk_vs_stop`. The book returns to this point later; for now the task is simply to transform `streetlights` into `walk_vs_stop` with a neural network.

> A neural network as a black box that turns streetlights into walk_vs_stop
>
> <img src="pictures/chapter 6/picture 6.9.png" alt="Figure 6.9" width="600"/>


---
## **5. Building a Neural Network**

The following cell trains a network on **only the first streetlight pattern**. It should look familiar. The weights start at `[0.5, 0.48, -0.7]`, and the learning rate is `alpha = 0.1`. The input is the first row, `[1,0,1]`, and the goal prediction is the first entry of `walk_vs_stop`, which is 0 (stop). In each of the 20 iterations the network:

1. makes a prediction with a dot product (`input.dot(weights)`),
2. computes the squared `error`,
3. computes `delta = prediction - goal_prediction`,
4. updates the weights with `weights = weights - (alpha * (input * delta))`,
5. prints the error and the prediction.

The `dot` function performs a dot product (a weighted sum) between two vectors, as in Chapter 3.

In [3]:
import numpy as np
weights = np.array([0.5,0.48,-0.7])
alpha = 0.1

streetlights = np.array( [ [ 1, 0, 1 ],
                           [ 0, 1, 1 ],
                           [ 0, 0, 1 ],
                           [ 1, 1, 1 ],
                           [ 0, 1, 1 ],
                           [ 1, 0, 1 ] ] )

walk_vs_stop = np.array( [ 0, 1, 0, 1, 1, 0 ] )

input = streetlights[0]
goal_prediction = walk_vs_stop[0]

for iteration in range(20):
    prediction = input.dot(weights)
    error = (goal_prediction - prediction) ** 2
    delta = prediction - goal_prediction
    weights = weights - (alpha * (input * delta))

    print("Error:" + str(error) + " Prediction:" + str(prediction))

Error:0.03999999999999998 Prediction:-0.19999999999999996
Error:0.025599999999999973 Prediction:-0.15999999999999992
Error:0.01638399999999997 Prediction:-0.1279999999999999
Error:0.010485759999999964 Prediction:-0.10239999999999982
Error:0.006710886399999962 Prediction:-0.08191999999999977
Error:0.004294967295999976 Prediction:-0.06553599999999982
Error:0.002748779069439994 Prediction:-0.05242879999999994
Error:0.0017592186044416036 Prediction:-0.04194304000000004
Error:0.0011258999068426293 Prediction:-0.03355443200000008
Error:0.0007205759403792803 Prediction:-0.02684354560000002
Error:0.0004611686018427356 Prediction:-0.021474836479999926
Error:0.0002951479051793508 Prediction:-0.01717986918399994
Error:0.00018889465931478573 Prediction:-0.013743895347199997
Error:0.00012089258196146188 Prediction:-0.010995116277759953
Error:7.737125245533561e-05 Prediction:-0.008796093022207963
Error:4.951760157141604e-05 Prediction:-0.007036874417766459
Error:3.169126500570676e-05 Prediction:-0.0

NumPy offers one more convenience that Chapter 3 did not cover: **elementwise** operations on vectors. The next cell, which the book presents in a side box, shows four of them:

- `a*b`: elementwise multiplication of two vectors,
- `a+b`: elementwise addition of two vectors,
- `a * 0.5`: vector-scalar multiplication,
- `a + 0.5`: vector-scalar addition.

When a `+` is put between two vectors, it does what one expects: it adds them together. Running the cell prints `[0 2 4 3]`, `[2 3 4 4]`, `[0.  0.5 1.  0.5]`, and `[0.5 1.5 2.5 1.5]`. Apart from these operators and the new dataset, the network is the same as those built in the earlier chapters.

In [4]:
import numpy as np

a = np.array([0,1,2,1])
b = np.array([2,2,2,3])
print(a*b)
print(a+b)
print(a * 0.5)
print(a + 0.5)

[0 2 4 3]
[2 3 4 4]
[0.  0.5 1.  0.5]
[0.5 1.5 2.5 1.5]


---
## **6. Learning the Whole Dataset**

**The neural network has been learning only one streetlight. Don't we want it to learn them all?**

So far, each network has modeled a single training example (one `input -> goal_pred` pair). A network that tells whether it is safe to cross the street must know more than one streetlight, so it is trained on **all the streetlights at once**. The following cell wraps the learning step in a second loop over the rows (`row_index`). For every row it selects the input and the goal, predicts, computes the error, accumulates it in `error_for_all_lights`, computes `delta`, and updates the weights immediately. After each pass through the six rows, the total error is printed.

The printed output of the book shows only the lines starting with `Error`: the first two values are about 2.656 and 0.963, and the last two are about 0.000614 and 0.000534. The cell also prints a `Prediction` line for every example, so its output is longer than the excerpt in the book. The two lines `input = streetlights[0]` and `goal_prediction = walk_vs_stop[0]` before the loop are kept as printed; they are overwritten inside the loop.

In [5]:
import numpy as np

weights = np.array([0.5,0.48,-0.7])
alpha = 0.1

streetlights = np.array( [[ 1, 0, 1 ],
                          [ 0, 1, 1 ],
                          [ 0, 0, 1 ],
                          [ 1, 1, 1 ],
                          [ 0, 1, 1 ],
                          [ 1, 0, 1 ] ] )

walk_vs_stop = np.array( [ 0, 1, 0, 1, 1, 0 ] )

input = streetlights[0]
goal_prediction = walk_vs_stop[0]

for iteration in range(40):
    error_for_all_lights = 0
    for row_index in range(len(walk_vs_stop)):
        input = streetlights[row_index]
        goal_prediction = walk_vs_stop[row_index]

        prediction = input.dot(weights)

        error = (goal_prediction - prediction) ** 2
        error_for_all_lights += error

        delta = prediction - goal_prediction
        weights = weights - (alpha * (input * delta))
        print("Prediction:" + str(prediction))
    print("Error:" + str(error_for_all_lights) + "\n")

Prediction:-0.19999999999999996
Prediction:-0.19999999999999996
Prediction:-0.5599999999999999
Prediction:0.6160000000000001
Prediction:0.17279999999999995
Prediction:0.17552
Error:2.6561231104

Prediction:0.14041599999999999
Prediction:0.3066464
Prediction:-0.34513824
Prediction:1.006637344
Prediction:0.4785034751999999
Prediction:0.26700416768
Error:0.9628701776715985

Prediction:0.213603334144
Prediction:0.5347420299776
Prediction:-0.26067345110016
Prediction:1.1319428845096962
Prediction:0.6274723921901568
Prediction:0.25433999330650114
Error:0.5509165866836797

Prediction:0.20347199464520088
Prediction:0.6561967149569552
Prediction:-0.221948503950995
Prediction:1.166258650532124
Prediction:0.7139004922542389
Prediction:0.21471099528371604
Error:0.36445836852222424

Prediction:0.17176879622697283
Prediction:0.7324724146523222
Prediction:-0.19966478845083285
Prediction:1.1697769945341199
Prediction:0.7719890116601171
Prediction:0.17297997428859369
Error:0.2516768662079895

Predictio

---
## **7. Full, Batch, and Stochastic Gradient Descent**

**Stochastic gradient descent updates weights one example at a time.**

Learning one example at a time, as in the previous cell, is a variant of gradient descent called **stochastic gradient descent**. It is one of a handful of methods that can learn an entire dataset. It predicts and updates the weights for each training example separately: it takes the first streetlight, tries to predict it, calculates the `weight_delta`, and updates the weights; then it moves on to the second streetlight, and so on. It cycles through the dataset many times until it finds a weight configuration that works well for all the examples.

| Variant | When the weights are updated |
|---|---|
| **Stochastic gradient descent** | after **each single** training example |
| **(Full) gradient descent** (also called average or full gradient descent, as introduced in Chapter 4) | **once per dataset**: the average `weight_delta` over the entire dataset is computed first, and the weights change only when this full average is available |
| **Batch gradient descent** | after **n examples**: a compromise between the two, in which a *batch size* (typically between 8 and 256) is chosen, and the weights are updated after each batch |

Batch gradient descent is covered in more detail later in the book. For now, the point is that the previous cell created a network that learns the whole streetlights dataset by training on each example, one at a time.

---
## **8. Neural Networks Learn Correlation**

**What did the last neural network learn?**

The single-layer network just trained takes a streetlight pattern and decides whether it is safe to cross. From the network's own perspective, however, it never knew it was processing streetlights. All it did was to determine **which of the three inputs correlated with the output**, and it did so correctly, as the final weights show.

> The final weights of the trained network: about .01 for the left input, 1.0 for the middle input, and about 0 (-.0) for the right input
>
> <img src="pictures/chapter 6/picture 6.10.png" alt="Figure 6.10" width="600"/>

The middle weight is very close to 1, while the far-left and far-right weights are very close to 0. At a high level, all the iterative and complex machinery of learning achieved something rather simple: it **identified the correlation between the middle input and the output**. The correlation is located wherever the weights were set to high numbers. Conversely, *randomness* with respect to the output was found at the far-left and far-right weights, whose values are near 0.

How did the network find the correlation? During gradient descent, each training example exerts either **up pressure** or **down pressure** on the weights. On average, the middle weight received more up pressure, and the other weights received more down pressure. The next questions are where this pressure comes from, and why it differs from weight to weight.

---
## **9. Up and Down Pressure**

**It comes from the data.**

Each node tries on its own to predict the output from its input, and for the most part it ignores the other nodes. The only "cross communication" is that **all three weights share the same error measure**. The weight update is nothing more than this shared error multiplied by each weight's own input.

This is a key reason why neural networks learn: **error attribution**. Given a shared error, the network has to find out which weights contributed to it (so that they can be adjusted) and which did not (so that they can be left alone).

> Training data (left) and the pressure each example puts on each weight (right): + pushes toward 1, - pushes toward 0, 0 means no pressure
>
> <img src="pictures/chapter 6/picture 6.11.png" alt="Figure 6.11" width="600"/>

Consider the first training example, `[1, 0, 1] -> 0`:

- The middle input is 0, so the middle weight is **completely irrelevant** for this prediction. Whatever its value, it is multiplied by 0. Any error in this example, whether the prediction is too high or too low, can therefore be attributed only to the far-left and far-right weights.
- The network should predict 0, but two inputs are 1, which creates error and drives those two weights **toward 0**.

The Weight Pressure table summarizes the effect of every example on every weight. A **+** indicates pressure toward 1, a **-** indicates pressure toward 0, and a **0** indicates no pressure, because the input of that example is 0 and the weight is not changed. The far-left weight has two negatives and one positive, so on average it moves toward 0. The middle weight has three positives, so on average it moves toward 1.

Each weight tries to compensate for the error. In the first example, the far-left and far-right inputs are **discorrelated** with the desired output, so those weights are pushed down. The same happens throughout all six examples: correlation is rewarded with pressure toward 1, and discorrelation is penalized with pressure toward 0. On average, the correlation between the middle input and the output becomes the dominant predictive force, that is, the heaviest weight in the weighted average of the inputs, and the network becomes quite accurate.

> **Bottom line.** The prediction is a weighted sum of the inputs. The learning algorithm rewards inputs that correlate with the output with upward pressure on their weight and penalizes inputs that are discorrelated with the output with downward pressure. The weighted sum finds the perfect correlation between input and output by weighting the decorrelated inputs to 0.

The author admits that a mathematician may cringe: "upward pressure" and "downward pressure" are not precise mathematical expressions, and they have edge cases in which the logic does not hold (two of them follow). Still, this is an extremely valuable approximation. It allows one to set aside the complexity of gradient descent and simply remember that **learning rewards correlation with larger weights** or, more generally, that **learning finds correlation between two datasets**.

---
## **10. Edge Case: Overfitting**

**Sometimes correlation happens accidentally.**

Look again at the first training example. Suppose the far-left weight were 0.5 and the far-right weight were -0.5. Their contributions would cancel, and the prediction would be 0, so the network would predict this example perfectly. Yet it would not have learned anything that works in the real world: those weights would fail on other streetlights. This phenomenon is called **overfitting**.

> **Deep learning's greatest weakness: overfitting.** The error is shared among all the weights. If a particular configuration of weights *accidentally* creates a perfect correlation between the prediction and the output (such that `error == 0`) without giving the heaviest weight to the best inputs, the neural network **stops learning**.

If it were not for the other training examples, this flaw would cripple the network. The second training example, however, pushes the far-right weight upward while leaving the far-left weight untouched, which breaks the equilibrium that had stopped the learning in the first example. As long as the network is not trained exclusively on the first example, the remaining examples help it escape such edge-case configurations that exist for any single example.

This point is very important. Neural networks are so flexible that they can find many different weight configurations that predict a *subset* of the training data correctly. If this network were trained only on the first two examples, it would probably stop at a point where it fails on the other examples. In essence, it would have **memorized** those two examples instead of finding the correlation that **generalizes** to any possible streetlight configuration, and it could fail on a streetlight that was not in the training data.

> **Key takeaway.** The greatest challenge in deep learning is convincing the network to **generalize** instead of just **memorize**. This theme returns later in the book.

---
## **11. Edge Case: Conflicting Pressure**

**Sometimes correlation fights itself.**

The Weight Pressure table is shown again. Look at the far-right column: it seems to contain as many upward as downward pressure moments (three of each). Nevertheless, the network correctly pushes this weight down to 0, so the downward pressure moments must somehow be stronger. How does that work?

> The Weight Pressure table again (the same figure as before): the far-right column has three + and three -
>
> <img src="pictures/chapter 6/picture 6.11.png" alt="Figure 6.11" width="600"/>

The explanation is that the **left and middle weights have enough signal to converge on their own**. The left weight falls to 0, and the middle weight moves toward 1. As the middle weight grows, the error on the positive examples keeps decreasing, but as the weights approach their optimal positions, the discorrelation of the far-right weight becomes more and more apparent.

In the extreme case, the left and middle weights are exactly 0 and 1. If the right weight is above 0, the network predicts too high; if it is below 0, the network predicts too low. When other nodes learn, they **absorb part of the error**, that is, part of the correlation. They give the network a moderate predictive power, which reduces the error, and the remaining weights then only adjust to predict whatever is left. Because the middle weight has a consistent signal (a 1:1 relationship between the middle input and the output), it absorbs the correlation entirely. The error on the examples that should be 1 becomes very small, while the error on the examples that should be 0 stays large, and this imbalance keeps pushing the weights downward. (The book words the last step as a push on the *middle* weight, although the weight being silenced in this discussion is the far-right one.)

### **11.1 It Doesn't Always Work Out Like This**

In some ways, the network was lucky. If the middle node had not been so perfectly correlated, it might have struggled to silence the far-right weight. This is where **regularization** comes in, a topic announced for later in the book. Regularization forces weights with conflicting pressure toward 0. The motivation is that a weight with equal upward and downward pressure is of no use in either direction. Regularization says that only weights with really strong correlation may stay on, and everything else is silenced because it only contributes noise. It resembles natural selection, and as a side effect the network would train faster (in fewer iterations), because the far-right weight, which is not definitively correlated, would immediately be driven toward 0. Without regularization, as in the training above, the network learns that the far-right input is useless only after the left and middle weights have found their patterns.

### **11.2 A Dataset Without Any Correlation**

If networks look for correlation between an input column and the output column, what happens with the following dataset?

> A new dataset in which no input column correlates with the output, and its Weight Pressure table
>
> <img src="pictures/chapter 6/picture 6.12.png" alt="Figure 6.12" width="600"/>

Here **no input column correlates with the output column**. Every weight receives as much upward pressure as downward pressure. Previously, a weight with both kinds of pressure could be rescued, because other nodes started to solve either the positive or the negative predictions and thereby drew the balanced node up or down. In this dataset, all inputs are balanced equally between positive and negative pressure, so that mechanism cannot help. This dataset is a real problem for the network, and the question is what to do about it.

---
## **12. Learning Indirect Correlation**

**If your data doesn't have correlation, create intermediate data that does!**

### **12.1 Networks Search for Correlation Between Layers**

Earlier, a neural network was described as an instrument that searches for correlation between an input and an output *dataset*. To be more exact, a network searches for correlation between its input and output **layers**. The input layer is set to individual rows of the input data, and the network is trained so that the output layer equals the output dataset. Oddly enough, the network does not know anything about the data; it only looks for correlation between the input layer and the output layer.

> The single-layer network again (the same figure as in Section 8): the weights connect an input layer directly to the output layer
>
> <img src="pictures/chapter 6/picture 6.10.png" alt="Figure 6.10" width="600"/>

The new streetlights dataset has **no correlation between its input and output**. The solution is simple: use **two of these networks**. The first one creates an intermediate dataset that has limited correlation with the output, and the second one uses this limited correlation to predict the output correctly.

> Because the input dataset does not correlate with the output dataset, the input is used to create an *intermediate* dataset that **does** correlate with the output. It is a bit like cheating.

### **12.2 Creating Correlation**

The new network consists of **two networks stacked on top of each other**. The middle layer of nodes, `layer_1`, represents the **intermediate dataset**. The goal is to train the network so that, although there is no correlation between `layer_0` and `layer_2` (the input and output datasets), the `layer_1` dataset that is *created from* `layer_0` does correlate with `layer_2`.

> Two networks stacked: layer_0 (input), layer_1 (the intermediate data), and layer_2 (walk/stop), connected by weights_0_1 and weights_1_2
>
> <img src="pictures/chapter 6/picture 6.13.png" alt="Figure 6.13" width="600"/>

This network is still just a **function**: a collection of weights arranged in a particular way. Gradient descent still works, because one can compute how much each weight contributes to the error and adjust it to reduce the error to 0. That is exactly what is done next.

### **12.3 Stacking Neural Networks: A Review**

Chapter 3 briefly mentioned stacked networks. The prediction occurs just as one would expect: the output of the lower network (`layer_0` to `layer_1`) is the input of the upper network (`layer_1` to `layer_2`), and each network makes its prediction exactly as before.

Learning is where the existing knowledge helps. If the lower weights are ignored and the values of `layer_1` are treated as a training set, the upper half of the network (`layer_1` to `layer_2`) is just like the networks trained in the preceding chapter, and all the same learning logic applies.

What is not yet clear is how to update the weights between `layer_0` and `layer_1`. What error measure should they use? As Chapter 5 explained, the cached and normalized error measure is called **`delta`**. The open question is therefore how to obtain the `delta` values at `layer_1`, so that `layer_1` can help `layer_2` make accurate predictions.

---
## **13. Backpropagation: Long-Distance Error Attribution**

### **13.1 The Weighted Average Error**

The prediction from `layer_1` to `layer_2` is a **weighted average** of the values in `layer_1`. If `layer_2` is too high by some amount x, which values in `layer_1` are responsible? The ones with **higher weights** (`weights_1_2`) contributed more, and those with **lower weights** contributed less. In the extreme case where a weight from `layer_1` to `layer_2` is zero, that `layer_1` node caused none of the error.

This is simple almost to the point of being funny. The weights from `layer_1` to `layer_2` describe exactly how much each `layer_1` node contributes to the `layer_2` prediction, and therefore also exactly how much each node contributes to the `layer_2` error. To obtain the `delta` at `layer_1` from the `delta` at `layer_2`, **multiply it by the respective weights**. It is the prediction logic run in reverse. This process of moving the `delta` signal around is called **backpropagation**.

> Backpropagation with made-up numbers: the layer_2 delta +0.25 is passed back through the weights 0.0, 0.5, 1.0, and -1.0 and gives layer_1 deltas of 0.0, 0.125, 0.25, and -0.25
>
> <img src="pictures/chapter 6/picture 6.14.png" alt="Figure 6.14" width="600"/>

In the figure, the `layer_2` delta is +0.25 (defined there as `goal_prediction - prediction`), and the weights from `layer_1` to `layer_2` are made-up values: 0.0, 0.5, 1.0, and -1.0. Multiplying the delta by each weight gives the `layer_1` deltas 0.0, 0.125, 0.25, and -0.25.

### **13.2 Why Does This Work? The Weighted Average Delta**

In the network of Chapter 5, the variable `delta` told a node the direction and the amount by which its value should change next time. Backpropagation lets one say: "If this node should be x higher, then each of these four previous nodes needs to be higher or lower by `x*weights_1_2`, because these weights amplified the prediction by `weights_1_2` times."

Used in reverse, the `weights_1_2` matrix **amplifies the error by the appropriate amount**, so it becomes clear how far each `layer_1` node should move up or down. Once this is known, each weight matrix can be updated as before: for every weight, multiply its output `delta` by its input value and adjust the weight by that amount (scaled with `alpha`, if desired).

The book repeats the same figure at this point:

> The same figure as above: layer_1 deltas are weighted versions of the layer_2 delta
>
> <img src="pictures/chapter 6/picture 6.14.png" alt="Figure 6.14" width="600"/>


---
## **14. Linear vs. Nonlinear**

**This is probably the hardest concept in the book. Let's take it slowly.**

One more piece is needed before this network can train. The book approaches it from two sides: first it shows **why the network is currently broken**, and afterwards what the missing piece does to fix it.

### **14.1 Two Multiplications Are Just One Multiplication**

Start with simple algebra. For any two multiplications, the same result can be reached with a single multiplication:

| Two multiplications | One multiplication |
|---|---|
| `1 * 0.25 * 0.9 = 0.225` | `1 * 0.225 = 0.225` |

(The book prints a first example, `1 * 10 * 2 = 100` and `5 * 20 = 100`, whose left side is a slip, since 1 * 10 * 2 is 20, not 100. The idea is shown correctly by the example in the table.)

This is bad news for stacked networks. Consider the two networks in the figure, each evaluated on two training examples, one with input 1.0 and one with input -1.0. In the three-layer network, the weight from the input to the middle node is 0.25 and the weight from the middle node to the output is 0.9, so the middle node takes the values 0.25 and -0.25, and the output is 0.225 and -0.225. The two-layer network with the single weight 0.225 produces exactly the same outputs.

> A three-layer network (weights 0.25 and 0.9) and a two-layer network (weight 0.225) give identical outputs for the inputs 1.0 and -1.0
>
> <img src="pictures/chapter 6/picture 6.15.png" alt="Figure 6.15" width="600"/>

The bottom line: **for any three-layer network, there is a two-layer network with identical behavior.** Stacking two neural networks, as they are known so far, does not add any power. Two consecutive weighted sums are just a more expensive version of one weighted sum.

### **14.2 Why the Neural Network Still Doesn't Work**

If the three-layer network were trained as it stands, it would not converge.

> **Problem.** For any two consecutive weighted sums of the input, there exists a single weighted sum with exactly identical behavior. Anything the three-layer network can do, the two-layer network can also do.

The reason becomes clear from the viewpoint of correlation. Each of the four middle nodes has a weight coming from every input and therefore *subscribes* to a certain amount of correlation with each input node. If a weight is 1.0, the middle node subscribes to exactly 100% of that input's movement: if the input rises by 0.3, the middle node follows. If the weight is 0.5, the node subscribes to exactly 50% of that movement.

> The three-layer network again (the same architecture as in Section 12): each middle node is only a mixture of the inputs
>
> <img src="pictures/chapter 6/picture 6.13.png" alt="Figure 6.13" width="600"/>

The only way for a middle node to escape the correlation with one input is to subscribe to additional correlation from another input. Nothing new is contributed. The middle nodes cannot add anything to the conversation, because they have **no correlation of their own**; they are only more or less correlated with various input nodes. In the new dataset, no input correlates with the output, so the middle layer merely mixes together correlations that are already useless.

What is needed is a middle layer that can **selectively** correlate with the input: sometimes it correlates with an input, and sometimes it does not. That gives the middle layer a correlation of its own. Instead of always being x% correlated with one input and y% correlated with another, a node can be x% correlated with one input *only when it wants to be*, and not correlated at all at other times. This is called **conditional correlation**, or **sometimes correlation**.

---
## **15. The Secret to Sometimes Correlation**

**Turn off the node when the value would be below 0.**

This may sound too simple to work. Normally, when a node's value drops below 0, the node still has the same correlation with its input as always; its value just happens to be negative. If the node is **turned off (set to 0) whenever it would be negative**, then it has **zero correlation with any input** while it is negative.

As a consequence, a node can pick and choose when it is correlated with something. For example, it can express: "Be perfectly correlated with the left input, but only when the right input is turned off." This is achieved with a weight of 1.0 from the left input and a hugely negative weight from the right input. When both inputs are on, the node is 0 all the time; when only the left input is on, the node takes on the value of the left input. This was impossible before, when the middle node was either always correlated with an input or never correlated. Now it can be **conditional**: it can speak for itself.

> **Solution.** By turning off any middle node whenever it would be negative, the network is allowed to subscribe *sometimes* to the correlation of various inputs. This is impossible for two-layer networks, which is how the extra power of three-layer networks comes about.

The technical term for the logic "if the node would be negative, set it to 0" is **nonlinearity**. Without this tweak, the network is **linear**: its output layer can only choose from the same correlation it had in the two-layer network, subscribing to pieces of the input layer, so it cannot solve the new streetlights dataset.

There are many kinds of nonlinearities. The one used here is, in many cases, the best one, and it is also the simplest. It is called **`relu`**.

The author adds a remark on terminology. Most other books and courses state that consecutive matrix multiplication is a *linear transformation*, a view he finds unintuitive and one that makes it harder to see what nonlinearities contribute and why one is chosen over another (a topic for later). In that formulation, without the nonlinearity, two matrix multiplications might as well be one. His own explanation is not the most concise, but it is meant to be an intuitive account of why nonlinearities are needed.

---
## **16. A Quick Break**

**That last part probably felt a little abstract, and that's totally OK.**

The earlier chapters rested on simple algebra, so everything was grounded in fundamentally simple tools. This chapter now builds on the premises established there. Two big lessons are involved:

1. *Earlier lesson:* one can compute the relationship between the error and any one of the weights, which shows how changing the weight changes the error, and this can be used to reduce the error to 0.
2. *Lesson of this chapter:* adjusting the weights to reduce the error over a series of training examples ultimately **searches for correlation between the input and the output layers**. If no correlation exists, the error will never reach 0.

The second lesson is the bigger one, and it largely lets one put the first out of mind: it is no longer necessary to think about how the error depends on each weight; the focus is now on correlation. The general advice is that nobody can think about everything at once. When a lesson is a more concise summary (a higher abstraction) of many granular lessons, it is fine to set the granular ones aside and concentrate on the higher summary.

The author compares this to a professional swimmer, cyclist, or baseball player: a great swing is the culmination of thousands of small lessons, but the player does not think about all of them at the plate, because the actions have become fluid and even subconscious. Studying these mathematical concepts is the same. Neural networks look for correlation between input and output; one no longer has to worry about *how*, only to know that it happens. The chapter now builds on that idea, and the reader is encouraged to relax and trust what has already been learned.

---
## **17. Your First Deep Neural Network**

**Here's how to make the prediction.**

The following cell initializes the weights and performs a forward propagation. (In the book, the new code is printed in bold: the definition of `relu`, the second weight matrix `weights_1_2`, and the computation of `layer_1` and `layer_2`.) It differs from the earlier two-layer code as follows:

- `relu(x)` is defined as `(x > 0) * x`: it **sets all negative numbers to 0** and leaves positive numbers unchanged.
- There are now **two sets of weights** that connect the three layers, `weights_0_1` (3 x `hidden_size`) and `weights_1_2` (`hidden_size` x 1), both initialized randomly (the expression `2*np.random.random(...) - 1` spreads the values over the range from -1 to 1). With `hidden_size = 4`, the middle layer has four nodes.
- `walk_vs_stop = np.array([[ 1, 1, 0, 0]]).T` is a column of four targets, matching the four rows of `streetlights`. These four rows and targets coincide with the correlation-free dataset of Section 11.2. The learning rate is now `alpha = 0.2`.
- The prediction goes through the layers one after the other. The input enters `layer_0`. Via `dot`, the signal travels up the weights from `layer_0` to `layer_1` (a weighted sum at each of the four `layer_1` nodes). These sums are then passed through `relu`, which turns all negative numbers into 0, and the result becomes the input of the next layer. A final weighted sum into the single output node gives `layer_2`.

> Forward propagation through the three-layer network: the layer_1 nodes are drawn with the relu symbol, which turns negative values into 0
>
> <img src="pictures/chapter 6/picture 6.16.png" alt="Figure 6.16" width="600"/>


In [6]:
import numpy as np

np.random.seed(1)

def relu(x):
    return (x > 0) * x

alpha = 0.2
hidden_size = 4

streetlights = np.array( [[ 1, 0, 1 ],
                          [ 0, 1, 1 ],
                          [ 0, 0, 1 ],
                          [ 1, 1, 1 ] ] )

walk_vs_stop = np.array([[ 1, 1, 0, 0]]).T

weights_0_1 = 2*np.random.random((3,hidden_size)) - 1
weights_1_2 = 2*np.random.random((hidden_size,1)) - 1

layer_0 = streetlights[0]
layer_1 = relu(np.dot(layer_0,weights_0_1))
layer_2 = np.dot(layer_1,weights_1_2)

---
## **18. Backpropagation in Code**

**You can learn the amount that each weight contributes to the final error.**

At the end of Chapter 5, the author asserted that memorizing the two-layer network code would be important, so that it can be recalled quickly when more advanced concepts are discussed. This is the moment when that matters. The following listing is the new learning code, and it is essential to recognize the parts that come from previous chapters; if they are not familiar, the advice is to go back to Chapter 5, memorize its code, and return.

### **18.1 The Learning Code**

*Notes on the cell below:*

- The listing in the book does not define `streetlights` and `walk_vs_stop`; it uses the ones from the forward-propagation listing of Section 17, which exist in the notebook namespace after running that cell, so the cell runs in sequence as printed. It also initializes the weights again.
- The book prints this listing with an indentation of three spaces per level, which is kept here.
- In this listing, `layer_2_delta` is computed as `goal - prediction`, and the weights are therefore *increased* (`+=`). The final program of Section 19 uses the opposite sign convention (`prediction - goal` and `-=`), which leads to the same updates.

Everything in the listing is fundamentally the same as in the previous chapter. The only truly new code consists of the function `relu2deriv` and the line that computes `layer_1_delta`. `relu2deriv(output)` returns 1 when `output > 0` and 0 otherwise, which is the **slope (the derivative) of `relu`**. Its purpose becomes clear below.

> Backpropagation in the network: the layer_2 delta is passed back through the weights to give the layer_1 deltas, and the nodes that relu turned off get no delta
>
> <img src="pictures/chapter 6/picture 6.17.png" alt="Figure 6.17" width="600"/>

The goal remains **error attribution**: finding out how much each weight contributed to the final error. In the two-layer network, a `delta` variable told how much higher or lower the output prediction should be. `layer_2_delta` is calculated in exactly the same way here. Once it is known how much the final prediction should move, one needs to know how much each middle (`layer_1`) node should move. These are effectively **intermediate predictions**. As soon as the delta at `layer_1` is known, the weight update is the same as before: for each weight, multiply its input value by its output `delta` and change the weight by that amount.

How are the `layer_1` deltas obtained?

1. Do the obvious thing and **multiply the output `delta` by each weight attached to it**. This weights how much each weight contributed to the error.
2. Consider one more factor: if `relu` set the output of a `layer_1` node to 0, that node did not contribute to the error, so its `delta` must also be 0. Multiplying each `layer_1` node by `relu2deriv` accomplishes this, since `relu2deriv` is either 1 or 0, depending on whether the `layer_1` value is greater than 0.

The learning loop runs for 60 iterations, accumulates `layer_2_error`, and prints it every 10th iteration (`iteration % 10 == 9`).

In [7]:
import numpy as np

np.random.seed(1)

def relu(x):
   return (x > 0) * x

def relu2deriv(output):
   return output>0

alpha = 0.2
hidden_size = 4

weights_0_1 = 2*np.random.random((3,hidden_size)) - 1
weights_1_2 = 2*np.random.random((hidden_size,1)) - 1

for iteration in range(60):
   layer_2_error = 0
   for i in range(len(streetlights)):
      layer_0 = streetlights[i:i+1]
      layer_1 = relu(np.dot(layer_0,weights_0_1))
      layer_2 = np.dot(layer_1,weights_1_2)

      layer_2_error += np.sum((layer_2 - walk_vs_stop[i:i+1]) ** 2)

      layer_2_delta = (walk_vs_stop[i:i+1] - layer_2)
      layer_1_delta=layer_2_delta.dot(weights_1_2.T)*relu2deriv(layer_1)

      weights_1_2 += alpha * layer_1.T.dot(layer_2_delta)
      weights_0_1 += alpha * layer_0.T.dot(layer_1_delta)

   if(iteration % 10 == 9):
      print("Error:" + str(layer_2_error))

Error:0.6342311598444467
Error:0.35838407676317513
Error:0.0830183113303298
Error:0.006467054957103705
Error:0.0003292669000750734
Error:1.5055622665134859e-05


### **18.2 One Iteration of Backpropagation**

The book also walks through a single iteration in four steps, using the first streetlight only and a network with a hidden layer of **three** nodes (`hidden_size = 3`). Each step is shown in a figure that contains the code of that step. Several boxes of the book repeat the lines of the preceding steps; the cells below show only the lines that are **new** in each step, and they are meant to be run in order so that the variables carry over.

**Step 1: Initializing the network's weights and data.** The figure draws the network with three inputs, three hidden nodes (each carrying the `relu` symbol), and one prediction node. The code initializes the weights and the data (here under the names `lights` and `walk_stop`).

> Step 1 of one backpropagation iteration: initializing the network's weights and data
>
> <img src="pictures/chapter 6/picture 6.18.png" alt="Figure 6.18" width="600"/>


In [8]:
import numpy as np

np.random.seed(1)

def relu(x):
    return (x > 0) * x

def relu2deriv(output):
    return output>0
lights = np.array( [[ 1, 0, 1 ],
                    [ 0, 1, 1 ],
                    [ 0, 0, 1 ],
                    [ 1, 1, 1 ] ] )

walk_stop = np.array([[ 1, 1, 0, 0]]).T

alpha = 0.2
hidden_size = 3

weights_0_1 = 2*np.random.random(\
                          (3,hidden_size)) - 1
weights_1_2 = 2*np.random.random(\
                          (hidden_size,1)) - 1

**Step 2: Predict + compare.** The network makes a prediction for the first streetlight, `[1, 0, 1]`, and calculates the output error and the delta. In the figure, the three hidden nodes take the values 0, .13, and 0 (the `relu` has turned the other two off), the prediction is -.02, and the error is 1.04.

> Step 2: making a prediction and calculating the output error and delta
>
> <img src="pictures/chapter 6/picture 6.19.png" alt="Figure 6.19" width="600"/>


In [9]:
layer_0 = lights[0:1]
layer_1 = np.dot(layer_0,weights_0_1)
layer_1 = relu(layer_1)
layer_2 = np.dot(layer_1,weights_1_2)

error = (layer_2-walk_stop[0:1])**2

layer_2_delta=(layer_2-walk_stop[0:1])

**Step 3: Learn - backpropagating from `layer_2` to `layer_1`.** The delta of `layer_2` is multiplied by the transposed weights `weights_1_2.T`, and the result is multiplied by `relu2deriv(layer_1)`, which sets to 0 the delta of every node that `relu` turned off. In the figure, the two nodes that are off keep a delta of 0, and only the middle node receives a nonzero delta.

> Step 3: backpropagating the delta from layer_2 to layer_1
>
> <img src="pictures/chapter 6/picture 6.20.png" alt="Figure 6.20" width="600"/>

*Note:* the figure labels the middle hidden node with .13 (its value) and -.17 (its delta), and the delta of the output as 0.14. Running the printed code gives a `layer_2_delta` of about -1.02 (`layer_2 - walk_stop[0:1]`) and a middle `layer_1_delta` of about +0.165. The magnitude 0.17 agrees with the figure; its sign corresponds to the opposite convention `goal - prediction` (see the note in Section 18.1), and the 0.14 does not match the code.

In [10]:
layer_1_delta=layer_2_delta.dot(weights_1_2.T)
layer_1_delta *= relu2deriv(layer_1)

**Step 4: Learn - generating the `weight_deltas` and updating the weights.** With the deltas of both layers available, each weight delta is the transposed input of the layer multiplied by the delta of the layer above it, `weight_delta_1_2 = layer_1.T.dot(layer_2_delta)` and `weight_delta_0_1 = layer_0.T.dot(layer_1_delta)`. Each weight matrix is then reduced by `alpha` times its weight delta.

> Step 4: generating the weight_deltas and updating the weights
>
> <img src="pictures/chapter 6/picture 6.21.png" alt="Figure 6.21" width="600"/>


In [11]:
weight_delta_1_2 = layer_1.T.dot(layer_2_delta)
weight_delta_0_1 = layer_0.T.dot(layer_1_delta)

weights_1_2 -= alpha * weight_delta_1_2
weights_0_1 -= alpha * weight_delta_0_1

As this iteration shows, **backpropagation is about calculating deltas for the intermediate layers so that gradient descent can be performed**. To do so, the weighted average `delta` of `layer_2` is taken for `layer_1`, weighted by the weights between the two layers. Then the nodes that did not participate in the forward prediction are turned off (set to 0), because they cannot have contributed to the error.

---
## **19. Putting It All Together**

**Here's the self-sufficient program you should be able to run.**

The following cell contains the whole program: the imports and the seed, the functions `relu` and `relu2deriv` (returning `x` if `x > 0` and 0 otherwise, and 1 if the input is greater than 0 and 0 otherwise, respectively), the streetlight data, the random initialization of the two weight matrices, and the training loop. In this version, the delta is computed as `layer_2 - walk_vs_stop[i:i+1]` and the weights are updated with `-=`.

The book prints the following output. It shows the error after every 10th of the 60 iterations, and the error falls steadily to almost 0:

| Iteration | Error |
|---|---|
| 10 | 0.634231159844 |
| 20 | 0.358384076763 |
| 30 | 0.0830183113303 |
| 40 | 0.0064670549571 |
| 50 | 0.000329266900075 |
| 60 | 1.50556226651e-05 |

(The cell may display more decimal places because of floating-point formatting.)

In [12]:
import numpy as np

np.random.seed(1)

def relu(x):
   return (x > 0) * x

def relu2deriv(output):
   return output>0

streetlights = np.array( [[ 1, 0, 1 ],
                          [ 0, 1, 1 ],
                          [ 0, 0, 1 ],
                          [ 1, 1, 1 ] ] )

walk_vs_stop = np.array([[ 1, 1, 0, 0]]).T

alpha = 0.2
hidden_size = 4

weights_0_1 = 2*np.random.random((3,hidden_size)) - 1
weights_1_2 = 2*np.random.random((hidden_size,1)) - 1

for iteration in range(60):
   layer_2_error = 0
   for i in range(len(streetlights)):
      layer_0 = streetlights[i:i+1]
      layer_1 = relu(np.dot(layer_0,weights_0_1))
      layer_2 = np.dot(layer_1,weights_1_2)

      layer_2_error += np.sum((layer_2 - walk_vs_stop[i:i+1]) ** 2)

      layer_2_delta = (layer_2 - walk_vs_stop[i:i+1])
      layer_1_delta=layer_2_delta.dot(weights_1_2.T)*relu2deriv(layer_1)

      weights_1_2 -= alpha * layer_1.T.dot(layer_2_delta)
      weights_0_1 -= alpha * layer_0.T.dot(layer_1_delta)

   if(iteration % 10 == 9):
      print("Error:" + str(layer_2_error))

Error:0.6342311598444467
Error:0.35838407676317513
Error:0.0830183113303298
Error:0.006467054957103705
Error:0.0003292669000750734
Error:1.5055622665134859e-05


---
## **20. Why Do Deep Networks Matter?**

**What's the point of creating "intermediate datasets" that have correlation?**

Consider a dataset of pictures that are labeled as containing a cat or not, and a network that should take the pixel values and predict whether a cat is in the picture. Just as in the last streetlight dataset, **no individual pixel correlates with the presence of a cat**. Only certain *configurations* of pixels correlate with it, so a two-layer network might have a problem.

> A cat picture: no single pixel indicates a cat, only configurations of pixels do
>
> <img src="pictures/chapter 6/picture 6.22.png" alt="Figure 6.22" width="600"/>

This is the **essence of deep learning**. Deep learning is all about creating **intermediate layers (datasets)** in which each node represents the presence or absence of a different configuration of inputs. For the cat images, no individual pixel needs to correlate with the cat. Instead, the middle layer tries to identify different configurations of pixels that may or may not correlate with a cat, such as an ear, cat eyes, or cat hair. The presence of many cat-like configurations then gives the final layer the information (the correlation) it needs to predict the presence or absence of a cat.

The three-layer network can be extended by stacking more and more layers. Some neural networks have hundreds of layers, and each node plays its part in detecting a different configuration of the input data. The rest of the book is devoted to studying different phenomena within these layers, in order to explore the full power of deep neural networks.

As in Chapter 5, the author issues a challenge: **memorize the previous code**. It is necessary to be very familiar with every operation in it for the following chapters to be readable. The recommendation is not to go on until one can build a three-layer neural network from memory.

---
## **21. Summary**

- **The streetlight problem** is a toy problem for learning an entire dataset. The data is split into what is known (the light states) and what is wanted (stop/walk), and both are expressed as **matrices**: one example per row, one measured thing per column. Many matrices (A, B, C) can express the same **underlying pattern**, and 0/1 coding of stop/walk is a lossless representation.
- With NumPy, a matrix is an array of arrays, and the operations needed are the dot product and elementwise vector operations.
- A network can learn a whole dataset by repeating prediction and weight update for each example in turn (**stochastic gradient descent**). The alternatives are **(full) gradient descent**, which updates once per dataset using the average `weight_delta`, and **batch gradient descent**, which updates after a chosen batch size (typically 8 to 256) of examples.
- **Neural networks learn correlation.** All weights share one error measure, and each example pushes each weight up (toward 1) or down (toward 0), depending on whether the input correlates with the output. **Error attribution** decides which weights are adjusted, and an input that is 0 puts no pressure on its weight. The heaviest weight ends up on the correlated input.
- The up/down pressure picture has edge cases. **Overfitting** occurs when a weight configuration accidentally predicts a subset of the examples perfectly, so that learning stops without correlation having been found. The other training examples help the network to escape such traps, because the goal is to **generalize**, not to memorize. **Conflicting pressure** arises when a weight has equal pressure in both directions. It was resolved here only because the other weights absorbed the correlation; **regularization** (announced for later) addresses it more directly.
- If a dataset has **no correlation between inputs and outputs**, a single layer cannot solve it. Networks search for correlation between layers, so a **middle layer** can create an **intermediate dataset** that does correlate with the output. This stacking of two networks gives a three-layer network.
- **Backpropagation** computes the delta of the middle layer by multiplying the delta of the output layer by the weights between the layers (the prediction logic in reverse) and then updating each weight as before: output delta times input value (scaled by `alpha`).
- Without a **nonlinearity**, two consecutive weighted sums are equivalent to a single one, so a deeper network would add no power. Setting a node to 0 whenever it would be negative (**`relu`**) gives it **conditional ("sometimes") correlation** with its inputs, which is what makes three-layer networks more powerful than two-layer networks.
- In the code, `relu(x) = (x > 0) * x` is used in the forward pass, and its slope, `relu2deriv(output) = output > 0`, is used in the backward pass to **turn off the delta of nodes that were off**. One iteration consists of **predict and compare**, **backpropagate the delta from `layer_2` to `layer_1`**, and **generate the weight deltas and update the weights**. The final program trains the three-layer network on the four streetlights, and its error falls from about 0.63 (after 10 iterations) to about 0.000015 (after 60).
- **Deep learning** creates intermediate layers whose nodes detect configurations of the inputs (such as the ears or eyes of a cat), and more layers can be stacked on top. The next chapters build on the code of this chapter, so it should be memorized.